# 4 · Warm-start from the example model
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
Fine-tuning starts with existing weights and continues training on new simulated data. It is not guaranteed to converge faster or transfer to your microscope; a suitable new PSF and noise model are still necessary. Here the example uses the **same example PSF and parameters**, solely to demonstrate the mechanism.

**Why the helper?** In the checked-out DECODE training code, `model_init` is present in the YAML but is not passed to the training model loader. Setting it alone can silently start from scratch. `finetune.py` calls the upstream engine, loads the state dict strictly into the model after trainer setup, and starts a fresh optimizer/scheduler. This is not checkpoint resumption. Different architecture or incompatible weights will produce an error. Verify the terminal prints `WARM START`. The helper does not modify DECODE or the original model. This path is based on the checked-out implementation and must be checked against the installed version on the lab PC.

Edit `decode_walkthrough/config.yaml` for walkthrough settings, then rerun from the first cell. The example model’s full DECODE parameters remain in its own `param_run.yaml`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


In [ ]:
import decode.utils.param_io
import torch
import inspect
import importlib
try:
    trainer = importlib.import_module("decode.neuralfitter.train.live_engine")  # 0.10.2
except ModuleNotFoundError as exc:
    if exc.name != "decode.neuralfitter.train.live_engine": raise
    trainer = importlib.import_module("decode.neuralfitter.train.train")  # development checkout
print('Training module:', inspect.getfile(trainer))
print('setup_trainer signature:', inspect.signature(trainer.setup_trainer))
param = decode.utils.param_io.load_params(str(project_path(CFG['paths']['example_parameters'])))
param.InOut.calibration_file = str(project_path(CFG['paths']['example_calibration']).resolve())
param.InOut.checkpoint_init = None
train_cfg = CFG['example_training']
param.Hardware.device = ('cuda:0' if torch.cuda.is_available() else 'cpu') if train_cfg['device'] == 'auto' else train_cfg['device']
param.Hardware.device_simulation = param.Hardware.device
param.Hardware.num_worker_train = train_cfg['workers']
param.HyperParameter.batch_size = train_cfg['batch_size']
param.HyperParameter.epochs = train_cfg['epochs']
param.HyperParameter.pseudo_ds_size = train_cfg['training_samples']
param.TestSet.test_size = train_cfg['test_samples']
param.HyperParameter.opt_param.lr = train_cfg['warmstart_learning_rate']  # illustrative; requires tuning
print('Example weights:', EXAMPLE / 'model.pt')


In [ ]:
WRITE_CONFIG = CFG['example_training']['write_config']
if WRITE_CONFIG:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / 'warmstart_runs').mkdir(exist_ok=True)
    param.InOut.experiment_out = str((WORK / 'warmstart_runs').resolve())
    config_path = WORK / 'warmstart_params.yaml'
    if config_path.exists(): raise FileExistsError(f'Choose a fresh path: {config_path}')
    decode.utils.param_io.save_params(str(config_path), param)
    helper = project_path(CFG['paths']['warmstart_helper'])
    print('In a terminal with the DECODE environment activated, run:')
    weights = project_path(CFG["paths"]["example_model"])
    print(f'python "{helper}" --param "{config_path}" --weights "{weights}"')
else:
    print('Set example_training.write_config: true in config.yaml to create the config and display the command.')


The helper creates its own new run directory via DECODE. `model.pt` is a weights file; DECODE’s `checkpoint_init` expects a training checkpoint with optimizer and scheduler state, so do not point it at `model.pt`. Evaluate any new model on independent examples and real data.
